# 02 · Configurações 1 e 2: modelo sem contexto × RAG simples
Mesmo modelo (`Qwen2.5-1.5B-Instruct`, rodando localmente na GPU do Colab), mesmo pedido de JSON.
A única diferença é o contexto recuperado da base.

In [1]:
# Preparação do ambiente (Colab ou local)
import os, sys, subprocess
REPO_URL = "https://github.com/giuliano10/GS-GENENERATIVE-AI.git"
SUBPASTA = ""   # projeto na raiz do repositório

if "google.colab" in sys.modules:
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", REPO_URL, "/content/repo"], check=True)
    else:   # mesma sessão de outro notebook: atualiza o código
        subprocess.run(["git", "-C", "/content/repo", "pull", "-q"])
    os.chdir(os.path.join("/content/repo", SUBPASTA))
    # O Colab já traz torch, transformers, numpy etc. Instalamos SÓ o que falta, sem fixar
    # versões: reinstalar o torch com a sessão aberta quebra o "import torch".
    import importlib.util
    faltando = [pip for mod, pip in [("peft", "peft"), ("accelerate", "accelerate"),
                ("sentence_transformers", "sentence-transformers"), ("fastapi", "fastapi"),
                ("httpx", "httpx"), ("tabulate", "tabulate"), ("bs4", "beautifulsoup4"),
                ("lxml", "lxml")] if importlib.util.find_spec(mod) is None]
    # O torchao que vem no Colab é antigo e faz o peft recusar o import; o projeto não o usa.
    if importlib.util.find_spec("torchao") is not None:
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
    if faltando:
        print("Instalando:", faltando)
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltando],
                           capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stdout[-3000:], r.stderr[-3000:])
            raise RuntimeError("Falha no pip install (detalhes acima)")
    # Cada notebook do Colab roda numa máquina separada: os resultados passam de um notebook
    # para o outro por uma pasta no Google Drive.
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = "/content/drive/MyDrive/assistente-calcadas-resultados"
    os.makedirs(PERSIST, exist_ok=True)
    subprocess.run(["cp", "-r", f"{PERSIST}/.", "."])
    print("Resultados anteriores trazidos do Drive:", sorted(os.listdir(PERSIST)))
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Pasta do projeto:", os.getcwd())

Mounted at /content/drive
Resultados anteriores trazidos do Drive: ['avaliacao']
Pasta do projeto: /content/repo/assistente-acessibilidade-calcadas


In [2]:
# Confere se os arquivos dos notebooks anteriores chegaram
for arq in ["avaliacao/limiar.json"]:
    assert os.path.exists(arq), f"Falta {arq}: rode antes o notebook 01 até o fim (ele salva no Drive)."
print("ok")

ok


In [3]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma (use Ambiente de execução > Alterar tipo > T4 GPU)")

GPU: Tesla T4


In [4]:
from src.base_conhecimento import carregar_trechos
from src.avaliacao import carregar_jsonl
from src.busca import Buscador, carregar_limiar
from src.config import ARQ_TESTE
from src.gerador import carregar_modelo
from src.experimento import rodar
trechos = carregar_trechos(); teste = carregar_jsonl(ARQ_TESTE)
buscador = Buscador(trechos, modo="hibrido"); limiar = carregar_limiar()
modelo = carregar_modelo(com_lora=False)
print(modelo.nome, "| limiar", limiar)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

... (15 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen/Qwen2.5-1.5B-Instruct | limiar 0.6216


## Configuração 1 — sem contexto

In [5]:
sem_ctx = rodar("sem_contexto", teste, modelo)
for s in sem_ctx[:3]: print(s["pergunta"], "\n ->", s["bruto"][:300], "\n")

[sem_contexto] 10/60
[sem_contexto] 20/60
[sem_contexto] 30/60
[sem_contexto] 40/60
[sem_contexto] 50/60
[sem_contexto] 60/60
Qual a largura mínima da faixa livre da calçada em São Paulo? 
 -> {
  "resposta": "Deve ser igual ou maior à metade do diâmetro máximo dos veículos.",
  "fontes": ["Lei nº 13.428/2016"],
  "recusa": false
} 

Qual a inclinação transversal máxima permitida na faixa livre da calçada? 
 -> {
  "resposta": "A inclinação transversal máxima permitida na faixa livre da calçada é de 10%.",
  "fontes": ["Lei Municipal nº 23.465/2017"],
  "recusa": false
} 

Qual a largura mínima da faixa de serviço da calçada? 
 -> {
  "resposta": "Deve ser igual ou maior à metade do diâmetro máximo dos veículos.",
  "fontes": ["Lei nº 13.709/2018"],
  "recusa": false
} 



## Configuração 2 — RAG simples

In [6]:
rag = rodar("rag", teste, modelo, buscador, limiar)
for s in rag[:3]: print(s["pergunta"], "\n ->", s["bruto"][:300], "\n")

[rag] 10/60
[rag] 20/60
[rag] 30/60
[rag] 40/60
[rag] 50/60
[rag] 60/60
Qual a largura mínima da faixa livre da calçada em São Paulo? 
 -> {"resposta":"A largura mínima da faixa livre da calçada em São Paulo é de 1,20m (um metro e vinte centímetros).", "fontes":["SP-D59671-art26","SP-D59671-art24"],"recusa":false} 

Qual a inclinação transversal máxima permitida na faixa livre da calçada? 
 -> {"resposta":"O limite máximo de inclinação transversal na faixa livre da calçada é de 8,33% (oito vírgula trinta e três por cento).","fontes":["SP-D59671-art22"],"recusa":false} 

Qual a largura mínima da faixa de serviço da calçada? 
 -> {"resposta":"A largura mínima da faixa de serviço da calçada é de 70cm (setenta centímetros).","fontes":["SP-D59671-art26"],"recusa":false} 



In [7]:
import pandas as pd
from src.avaliacao import corrigir, placar
ids = {t["id"] for t in trechos}; fids = {t["fonte_id"] for t in trechos}
pd.DataFrame([placar([corrigir(c, s, ids, fids) for c, s in zip(teste, saidas)], nome)
              for nome, saidas in [("sem_contexto", sem_ctx), ("rag", rag)]])

,configuracao,n,acuracia_%,acerto_consta_%,recusa_correta_nao_consta_%,alucinacao_%,recusa_indevida_%,json_valido_%,citacao_correta_%,citacao_fora_contexto_%,latencia_media_s
0,sem_contexto,60,11.7,11.1,13.3,88.3,0.0,96.7,0.0,NaN,2.77
1,rag,60,68.3,73.3,53.3,25.0,8.9,96.7,68.3,3.3,2.47


In [8]:
# Salva os resultados no Google Drive (para o próximo notebook usar)
import shutil
FIXOS = {"casos_teste.jsonl", "lora_treino.jsonl", ".gitkeep"}
if "google.colab" in sys.modules:
    for pasta in ["avaliacao", "modelos"]:
        if os.path.exists(pasta):
            shutil.copytree(pasta, os.path.join(PERSIST, pasta), dirs_exist_ok=True,
                            ignore=lambda d, nomes: [n for n in nomes if n in FIXOS or n.endswith("_ckpt")])
    print("Salvo no Drive:", {p: sorted(os.listdir(os.path.join(PERSIST, p)))
                              for p in os.listdir(PERSIST) if os.path.isdir(os.path.join(PERSIST, p))})

Salvo no Drive: {'avaliacao': ['busca_recall.csv', 'limiar.json', 'limiar.png', 'respostas_rag.jsonl', 'respostas_sem_contexto.jsonl']}
